In [1]:
import os
import sys
from pathlib import Path

# Setup Colab environment
if 'google.colab' in sys.modules:
    repo_name = 'oc-hosp-surge-bn'
    repo_path = Path(f'/content/{repo_name}')

    if not repo_path.exists():
        !git clone https://github.com/AnhJoe/{repo_name}.git
    
    os.chdir(repo_path)
    !git pull origin main
    !apt-get install -y graphviz > /dev/null
    !pip install -r requirements.txt
# If local, install requirements
else:
    !apt-get install -y graphviz > /dev/null
    !pip install -r requirements.txt

# Find project root via src folder
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

# Configure path and working directory
if (root / "src").exists():
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    os.chdir(root)
    print(f"Project root: {root}")
else:
    print("Warning: 'src' folder not found.")

The system cannot find the path specified.


Project root: c:\Users\joetn\oc-ems-bayes-modeling


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import glob
import re
import gdown


# Analysis and modeling
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Project-specific EDA utilities
from src.eda import (
    choose_k_gap_rule,
    evaluate_kmeans_k,
    fit_kmeans,
    pca_project,
    plot_clusters_pca,
    plot_k_diagnostics,
)

# Global visualization styling
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "legend.title_fontsize": 9,
    "legend.frameon": True,
    "legend.borderpad": 0.3,
})

In [3]:
#| output: false

# This will download all required datasets from a public google drive folder into data/raw
target_root = root / "data" / "raw"
folder_id = '1hkw97vM9arXhGRFRMZXundV16FBBeD0H'

if not os.path.exists(target_root):
    os.makedirs(target_root, exist_ok=True)
    print(f"Created directory at: {target_root}")
    # Download into the absolute root path
    !gdown --folder {folder_id} -O {target_root}
else:
    print(f"Directory already exists at: {target_root}")

# Verify file presence in the root directory
print("Files in root data directory:", os.listdir(target_root))

Directory already exists at: c:\Users\joetn\oc-ems-bayes-modeling\data\raw
Files in root data directory: ['2017pddpivot.csv', '2018pddpivot.csv', '2019pddpivot.csv', '2020pddpivot.csv', '2021pddpivot.csv', '2022pddpivot.csv', '2023pddpivot.csv', '2024pddpivot.csv', 'ca-hcai-preventablehospitalizations-county_2005-2024.csv']


In [4]:
# 1. Read raw preventable dataset
hcai_path = os.path.join(target_root, 'ca-hcai-preventablehospitalizations-county_2005-2024.csv')
df_preventable_raw = pd.read_csv(hcai_path)

# 2. Filter for Orange County and Year >= 2017
df_hcai_oc = df_preventable_raw[
    (df_preventable_raw['County'] == 'Orange') & 
    (df_preventable_raw['Year'] >= 2017)
].copy()

# 3. Filter for PQI 91, 92, and 93
pqi_list = [91, 92, 93]
df_hcai_oc_filtered = df_hcai_oc[df_hcai_oc['PQI'].isin(pqi_list)].copy()

# 4. Retain only specific columns
columns_to_keep = ['Year', 'County', 'PQI', 'PQIDescription', 'RiskAdjRate_ICD10']
df_hcai_oc_final = df_hcai_oc_filtered[columns_to_keep]

df_hcai_oc_final.head(10)

,Year,County,PQI,PQIDescription,RiskAdjRate_ICD10
15519,2017,Orange,91,Acute Composite,232.4
15520,2018,Orange,91,Acute Composite,237.7
15521,2019,Orange,91,Acute Composite,213.6
15522,2020,Orange,91,Acute Composite,156.8
15523,2021,Orange,91,Acute Composite,135.1
15524,2022,Orange,91,Acute Composite,157.3
15525,2023,Orange,91,Acute Composite,183.9
15526,2024,Orange,91,Acute Composite,175.1
16699,2017,Orange,92,Chronic Composite,750.9
16700,2018,Orange,92,Chronic Composite,699


In [8]:
# Data Validation
print("--- Filter Validation ---")
# Check row counts to verify reduction
print(f"Total rows in raw statewide data: {len(df_preventable_raw)}")
print(f"Rows after Orange County filter: {len(df_hcai_oc)}")
print(f"Rows after Year >= 2017 & PQI 91-93 filters: {len(df_hcai_oc_filtered)}")

# Validation 1: Ensure only 'Orange' exists in the subset
unique_counties = df_hcai_oc_filtered['County'].unique()
assert len(unique_counties) == 1 and unique_counties[0] == 'Orange', f"Error: Multiple counties found: {unique_counties}"
print(f"Check 1 (County Integrity): Only {unique_counties[0]} present.")

# Validation 2: Verify Year range
year_min, year_max = df_hcai_oc_filtered['Year'].min(), df_hcai_oc_filtered['Year'].max()
print(f"Check 2 (Temporal Range): Range: {year_min} to {year_max}")

# Validation 3: PQI Completeness
# We expect 14 unique PQIs (Individual conditions + Composites)
unique_pqis = sorted(df_hcai_oc_filtered['PQI'].unique())
print(f"Check 3 (PQI Count): Found {len(unique_pqis)} unique PQIs: {unique_pqis}")

# Validation 4: Metric Availability (Risk-Adjusted Rate)
# Converting to numeric to check for 'S' (suppressed) or 'NaN' values
df_hcai_oc_filtered['RAR_Numeric'] = pd.to_numeric(df_hcai_oc_filtered['RiskAdjRate_ICD10'], errors='coerce')
missing_rar = df_hcai_oc_filtered['RAR_Numeric'].isnull().sum()
print(f"Check 4 (RAR Availability): Found {missing_rar} missing/suppressed numeric rates out of {len(df_hcai_oc_filtered)} rows.")

if missing_rar > 0:
    print("WARNING: Some RAR values are suppressed (likely due to low volume).")

--- Filter Validation ---
Total rows in raw statewide data: 17818
Rows after Orange County filter: 112
Rows after Year >= 2017 & PQI 91-93 filters: 24
Check 1 (County Integrity): Only Orange present.
Check 2 (Temporal Range): Range: 2017 to 2024
Check 3 (PQI Count): Found 3 unique PQIs: [np.int64(91), np.int64(92), np.int64(93)]
Check 4 (RAR Availability): Found 0 missing/suppressed numeric rates out of 24 rows.


In [9]:
# Map the specific PDD filenames (2017-2024)
pdd_filenames = {
    2017: '2017pddpivot.csv',
    2018: '2018pddpivot.csv',
    2019: '2019pddpivot.csv',
    2020: '2020pddpivot.csv',
    2021: '2021pddpivot.csv',
    2022: '2022pddpivot.csv',
    2023: '2023pddpivot.csv',
    2024: '2024pddpivot.csv'
}

# Dictionary to store the filtered Orange County DataFrames
pdd_oc_dict = {}

print("--- PDD Load & Filter Validation (Orange County) ---")

for year, filename in pdd_filenames.items():
    pdd_path = os.path.join(target_root, filename)
    
    try:
        df_raw = pd.read_csv(pdd_path)
        
        # Normalize headers: remove spaces and force uppercase
        df_raw.columns = df_raw.columns.str.strip().str.upper()
        
        # Identify which county column is present (COUNTY or COUNTY_NAME)
        # This handles the schema shift between 2017-2020 and 2021-2024
        county_col = None
        if 'COUNTY' in df_raw.columns:
            county_col = 'COUNTY'
        elif 'COUNTY_NAME' in df_raw.columns:
            county_col = 'COUNTY_NAME'
        
        if county_col:
            # Filter for ORANGE (standardized to uppercase)
            df_oc = df_raw[df_raw[county_col].str.upper().str.strip() == 'ORANGE'].copy()
            pdd_oc_dict[year] = df_oc
            print(f"Year {year}: {len(df_oc)} facilities loaded using column '{county_col}'.")
        else:
            print(f"Year {year}: ERROR - Neither 'COUNTY' nor 'COUNTY_NAME' found. Columns: {list(df_raw.columns[:5])}")

    except FileNotFoundError:
        print(f"Year {year}: File NOT FOUND at {pdd_path}")
    except Exception as e:
        print(f"Year {year}: Unexpected error: {e}")

--- PDD Load & Filter Validation (Orange County) ---
Year 2017: 32 facilities loaded using column 'COUNTY_NAME'.
Year 2018: 32 facilities loaded using column 'COUNTY_NAME'.
Year 2019: 32 facilities loaded using column 'COUNTY_NAME'.
Year 2020: 32 facilities loaded using column 'COUNTY_NAME'.
Year 2021: 32 facilities loaded using column 'COUNTY'.
Year 2022: 35 facilities loaded using column 'COUNTY'.
Year 2023: 33 facilities loaded using column 'COUNTY'.
Year 2024: 33 facilities loaded using column 'COUNTY'.


In [10]:
def check_id_consistency(y1, y2):
    # Get unique OSHPD IDs (more reliable than names)
    ids_y1 = set(pdd_oc_dict[y1]['OSHPD_ID'].unique())
    ids_y2 = set(pdd_oc_dict[y2]['OSHPD_ID'].unique())
    
    gained = ids_y2 - ids_y1
    lost = ids_y1 - ids_y2
    
    print(f"\n--- {y1} to {y2} ID Audit ---")
    print(f"Net Change: {len(ids_y2) - len(ids_y1)}")
    
    if gained:
        names = pdd_oc_dict[y2][pdd_oc_dict[y2]['OSHPD_ID'].isin(gained)]['FACILITY_NAME'].unique()
        print(f"Added IDs: {gained} ({names})")
    if lost:
        names = pdd_oc_dict[y1][pdd_oc_dict[y1]['OSHPD_ID'].isin(lost)]['FACILITY_NAME'].unique()
        print(f"Lost IDs:  {lost} ({names})")
    
    if not gained and not lost:
        print("Perfect Match: All facilities stayed consistent.")

# Run the audit
for yr in range(2017, 2024):
    check_id_consistency(yr, yr+1)


--- 2017 to 2018 ID Audit ---
Net Change: 0
Perfect Match: All facilities stayed consistent.

--- 2018 to 2019 ID Audit ---
Net Change: 0
Perfect Match: All facilities stayed consistent.

--- 2019 to 2020 ID Audit ---
Net Change: 0
Perfect Match: All facilities stayed consistent.

--- 2020 to 2021 ID Audit ---
Net Change: 0
Perfect Match: All facilities stayed consistent.

--- 2021 to 2022 ID Audit ---
Net Change: 3
Added IDs: {np.int64(301097), np.int64(304585), np.int64(304589)} (<StringArray>
[                   'ANAHEIM COMMUNITY HOSPITAL, LLC',
 'ENCOMPASS HEALTH REHABILITATION HOSPITAL OF TUSTIN',
                 'ALISO RIDGE BEHAVIORAL HEALTH, LLC']
Length: 3, dtype: str)

--- 2022 to 2023 ID Audit ---
Net Change: -2
Lost IDs:  {np.int64(301304), np.int64(304079)} (<StringArray>
['NEWPORT BAY HOSPITAL', 'ENCOMPASS HEALTH REHABILITATION HOSPITAL OF TUSTIN']
Length: 2, dtype: str)

--- 2023 to 2024 ID Audit ---
Net Change: 0
Perfect Match: All facilities stayed consistent.


In [ ]:
# Filter pdd next